# Agent 1 — Baseline v2 : mesurer le modèle de base AVANT fine-tuning

Mesure sur l'ensemble de **validation** (16 exemples), avec deux variantes :
- **brut** : la réponse du modèle telle quelle ;
- **brut + réparation** : on corrige automatiquement la fermeture des crochets JSON (l'erreur la plus fréquente du modèle brut).

Le fine-tuning devra faire mieux que *brut + réparation*, pas seulement mieux que *brut*.

**Avant de commencer** : *Exécution → Modifier le type d'exécution → GPU T4*.
Pour mesurer l'autre modèle : change `MODEL_ID` dans la cellule 4, puis *Exécution → Redémarrer la session* et relance tout.

In [ ]:
# Cellule 1 — installation
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
# Cellule 2 — vérifier le GPU
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Carte :", torch.cuda.get_device_name(0))

In [ ]:
# Cellule 3 — envoyer train.jsonl, val.jsonl et test.jsonl (depuis data/processed/)
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
# Cellule 4 — charger le modèle en 4-bit
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"      # autre modèle à comparer : "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model.eval()
print(f"Modèle chargé : {MODEL_ID} | mémoire GPU : {torch.cuda.memory_allocated()/1e9:.2f} Go")

In [ ]:
# Cellule 5 — fonction de génération (même protocole pour tous les modèles : température 0.7, graine fixée)
def generate(messages, max_new_tokens=1200):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.7, top_p=0.9,
            pad_token_id=tok.eos_token_id,
        )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Cellule 6 — données et références
import json, random
from collections import defaultdict

def load(path):
    return [json.loads(l) for l in open(path, encoding="utf-8")]

def answer_json(ex):                      # la réponse attendue (3e message) en dict
    return json.loads(ex["messages"][2]["content"])

def norm(s):
    return str(s).strip().casefold()

train = load("train.jsonl")
val = load("val.jsonl")
random.Random(0).shuffle(val)             # mélange pour couvrir plusieurs thèmes

# Catégories "officielles" = celles que contient le jeu d'entraînement
TRAIN_CATS = {norm(t["category"]) for ex in train for t in answer_json(ex)["tasks"]}
print(len(TRAIN_CATS), "catégories dans le train :", sorted(TRAIN_CATS))

# Référence de durée : moyenne du total de jours des réponses attendues pour une même description
tot = defaultdict(list)
for ex in val:
    tot[ex["messages"][1]["content"]].append(sum(t["estimate_days"] for t in answer_json(ex)["tasks"]))
REF_MEAN = {k: sum(v) / len(v) for k, v in tot.items()}
print(len(val), "exemples de validation,", len(REF_MEAN), "projets distincts")

In [ ]:
# Cellule 7 — les mesures
import re, statistics

REQUIRED = {"name", "description", "estimate_days", "category"}

def strip_fences(text):
    return re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())

def parse_output(raw, repair=False):
    text = strip_fences(raw)
    try:
        return json.loads(text)
    except Exception:
        if not repair:
            return None
    try:                                   # réparation : on referme proprement la liste puis l'objet
        return json.loads(text.rstrip().rstrip("}]").rstrip() + "}]}")
    except Exception:
        return None

def score(parsed, prompt):
    out = dict(valid=False, schema_ok=False, n_tasks=0, n_aberrant=0, n_cat_ok=0, rel_err=None)
    if parsed is None:
        return out
    out["valid"] = True
    tasks = parsed.get("tasks") if isinstance(parsed, dict) else None
    if not (isinstance(tasks, list) and tasks and all(isinstance(t, dict) and REQUIRED <= set(t) for t in tasks)):
        return out
    out["schema_ok"] = True
    out["n_tasks"] = len(tasks)
    days = [t["estimate_days"] for t in tasks]
    is_num = [isinstance(d, (int, float)) for d in days]
    out["n_aberrant"] = sum(1 for d, ok in zip(days, is_num) if not ok or not (1 <= d <= 15))
    out["n_cat_ok"] = sum(1 for t in tasks if norm(t["category"]) in TRAIN_CATS)
    ref = REF_MEAN.get(prompt)
    if ref and all(is_num):
        out["rel_err"] = abs(sum(days) - ref) / ref
    return out

def summarize(rows, repair):
    sc = [score(parse_output(r["raw"], repair), r["prompt"]) for r in rows]
    n = len(sc)
    ok = [s for s in sc if s["schema_ok"]]
    n_tasks = max(sum(s["n_tasks"] for s in sc), 1)
    rel = [s["rel_err"] for s in sc if s["rel_err"] is not None]
    return {
        "json_valid_rate": sum(s["valid"] for s in sc) / n,
        "schema_ok_rate": sum(s["schema_ok"] for s in sc) / n,
        "aberrant_estimate_rate": sum(s["n_aberrant"] for s in sc) / n_tasks,
        "category_conformity": sum(s["n_cat_ok"] for s in sc) / n_tasks,
        "median_total_rel_error": statistics.median(rel) if rel else None,
        "avg_tasks": (sum(s["n_tasks"] for s in ok) / len(ok)) if ok else None,
    }

In [ ]:
# Cellule 8 — génération sur l'ensemble de validation (compte ~10 minutes pour 16 exemples avec le 3B)
import time
N_EVAL = 16
torch.manual_seed(0)

rows = []
for ex in val[:N_EVAL]:
    prompt = ex["messages"][1]["content"]
    t0 = time.time()
    raw = generate(ex["messages"][:2])
    rows.append({"theme": ex["theme"], "prompt": prompt, "raw": raw, "seconds": round(time.time() - t0, 1)})
    print(f"[{len(rows)}/{N_EVAL}] {ex['theme'][:45]:45s} {rows[-1]['seconds']}s")

In [ ]:
# Cellule 9 — tableau de résultats
res = {"brut": summarize(rows, repair=False), "brut + réparation": summarize(rows, repair=True)}
keys = ["json_valid_rate", "schema_ok_rate", "aberrant_estimate_rate",
        "category_conformity", "median_total_rel_error", "avg_tasks"]
fmt = lambda v: "n/a" if v is None else f"{v:.2f}"

print(f"Modèle : {MODEL_ID} | {len(rows)} exemples de validation\n")
print(f"{'mesure':26s} {'brut':>8s} {'+ répar.':>9s}")
for k in keys:
    print(f"{k:26s} {fmt(res['brut'][k]):>8s} {fmt(res['brut + réparation'][k]):>9s}")
print("secondes par exemple :", round(sum(r["seconds"] for r in rows) / len(rows), 1))

In [ ]:
# Cellule 10 — sauvegarder (le fichier se télécharge sur ton PC)
tag = MODEL_ID.split("/")[-1].replace(".", "_")
name = f"baseline_{tag}_val{len(rows)}.json"
json.dump({"model": MODEL_ID, "split": "val", "summary": res, "rows": rows},
          open(name, "w", encoding="utf-8"), ensure_ascii=False, indent=2)
files.download(name)